In [ ]:
import pygame
import sys

# --- 1. CONFIGURATION DU JEU ---
pygame.init()
L_ECRAN, H_ECRAN = 800, 600
ecran = pygame.display.set_mode((L_ECRAN, H_ECRAN))
pygame.display.set_caption("Projet Jeu Isométrique - Programmation Multimédia")
horloge = pygame.time.Clock()

# Dimensions d'une tuile au sol
TILE_W = 64
TILE_H = 32

# --- 2. LA CARTE (GRID) ---
# 0 = Sol (Herbe), 1 = Mur (Obstacle)
carte = [
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 1, 1, 0, 1, 0, 1],
    [1, 0, 1, 0, 0, 1, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1]
]

# Position de départ du joueur sur la grille (x, y)
joueur_x, joueur_y = 1, 1

def cartesian_to_iso(x, y):
    """ Convertit les coordonnées de la grille en pixels écran """
    iso_x = (x - y) * (TILE_W // 2) + (L_ECRAN // 2)
    iso_y = (x + y) * (TILE_H // 2) + (H_ECRAN // 4)
    return iso_x, iso_y

def dessiner_tuile(x, y, type_tuile):
    """ Dessine une tuile ou un mur en fonction du type """
    px, py = cartesian_to_iso(x, y)
    
    # Points pour le losange du sol
    points = [
        (px, py + TILE_H // 2),
        (px + TILE_W // 2, py),
        (px + TILE_W, py + TILE_H // 2),
        (px + TILE_W // 2, py + TILE_H)
    ]
    
    if type_tuile == 1: # MUR (On dessine un bloc haut)
        # Face supérieure
        pygame.draw.polygon(ecran, (120, 120, 120), points) 
        # Côté gauche du bloc
        pygame.draw.polygon(ecran, (80, 80, 80), [
            (px, py + TILE_H // 2),
            (px + TILE_W // 2, py + TILE_H),
            (px + TILE_W // 2, py + TILE_H + 20),
            (px, py + TILE_H // 2 + 20)
        ])
        # Côté droit du bloc
        pygame.draw.polygon(ecran, (60, 60, 60), [
            (px + TILE_W, py + TILE_H // 2),
            (px + TILE_W // 2, py + TILE_H),
            (px + TILE_W // 2, py + TILE_H + 20),
            (px + TILE_W, py + TILE_H // 2 + 20)
        ])
    else: # SOL
        pygame.draw.polygon(ecran, (34, 139, 34), points) # Vert herbe
    
    # Bordure pour voir la grille
    pygame.draw.polygon(ecran, (20, 20, 20), points, 1)

def dessiner_joueur(x, y):
    """ Dessine le personnage sur la grille """
    px, py = cartesian_to_iso(x, y)
    # On dessine un cercle rouge pour le joueur (décalé vers le haut pour l'effet 3D)
    pygame.draw.circle(ecran, (255, 0, 0), (px + TILE_W // 2, py + TILE_H // 2 - 10), 10)
    # Petite ombre
    pygame.draw.ellipse(ecran, (0, 0, 0, 100), (px + TILE_W // 4, py + TILE_H // 2, TILE_W // 2, TILE_H // 2))

# --- 3. BOUCLE PRINCIPALE ---
continuer = True
while continuer:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            continuer = False
        
        # Gestion des touches pour déplacer le joueur
        if event.type == pygame.KEYDOWN:
            nouvel_x, nouvel_y = joueur_x, joueur_y
            if event.key == pygame.K_UP:    nouvel_y -= 1
            if event.key == pygame.K_DOWN:  nouvel_y += 1
            if event.key == pygame.K_LEFT:  nouvel_x -= 1
            if event.key == pygame.K_RIGHT: nouvel_x += 1
            
            # Vérification des collisions (ne pas marcher dans les murs)
            if 0 <= nouvel_x < len(carte[0]) and 0 <= nouvel_y < len(carte):
                if carte[nouvel_y][nouvel_x] == 0: # Si c'est du sol
                    joueur_x, joueur_y = nouvel_x, nouvel_y

    # --- RENDU ---
    ecran.fill((135, 206, 235)) # Fond bleu ciel
    
    # On parcourt la carte pour tout dessiner
    for y in range(len(carte)):
        for x in range(len(carte[0])):
            dessiner_tuile(x, y, carte[y][x])
            
            # Si le joueur est sur cette case, on le dessine MAINTENANT
            # (Cela permet la superposition correcte : Z-Ordering)
            if x == joueur_x and y == joueur_y:
                dessiner_joueur(x, y)

    pygame.display.flip()
    horloge.tick(30) # 30 images par seconde

pygame.quit()
sys.exit()

In [ ]:
import pygame

# --- INITIALISATION ---
pygame.init()
L, H = 800, 600
fenetre = pygame.display.set_mode((L, H))
pygame.display.set_caption("Mon Donjon Isométrique")
clock = pygame.time.Clock()

# Paramètres de la grille
TW, TH = 64, 32  # Taille des tuiles

# Couleurs "Design"
CIEL = (45, 52, 54)
HERBE = (38, 166, 91)
HERBE_BORD = (30, 130, 76)
MUR_TOP = (149, 165, 166)
MUR_SIDE = (99, 110, 114)
JOUEUR = (231, 76, 60) # Rouge corail

# --- LA CARTE ---
# 1 = Mur, 0 = Passage
labirynthe = [
    [1,1,1,1,1,1,1,1],
    [1,0,0,0,0,0,0,1],
    [1,0,1,1,0,1,0,1],
    [1,0,1,0,0,0,0,1],
    [1,0,0,0,1,1,0,1],
    [1,1,1,1,1,1,1,1]
]

perso_pos = [1, 1] # x, y

def get_screen_pos(gx, gy):
    """ Calcule la position x,y sur l'écran à partir de la grille """
    # Formule iso standard décalée pour centrer
    sx = (gx - gy) * (TW // 2) + (L // 2) - (TW // 2)
    sy = (gx + gy) * (TH // 2) + (H // 4)
    return sx, sy

def draw_block(x, y, is_wall):
    sx, sy = get_screen_pos(x, y)
    
    # Points du losange (le dessus)
    top_poly = [
        (sx + TW//2, sy), 
        (sx + TW, sy + TH//2), 
        (sx + TW//2, sy + TH), 
        (sx, sy + TH//2)
    ]
    
    if is_wall:
        # On dessine les côtés du cube pour l'effet 3D
        # Côté droit
        pygame.draw.polygon(fenetre, MUR_SIDE, [top_poly[1], top_poly[2], (sx+TW, sy+TH//2+20), (sx+TW//2, sy+TH+20)])
        # Côté gauche
        pygame.draw.polygon(fenetre, (45, 52, 54), [top_poly[3], top_poly[2], (sx, sy+TH//2+20), (sx+TW//2, sy+TH+20)])
        # Dessus
        pygame.draw.polygon(fenetre, MUR_TOP, top_poly)
    else:
        # Simple tuile d'herbe
        pygame.draw.polygon(fenetre, HERBE, top_poly)
        pygame.draw.polygon(fenetre, HERBE_BORD, top_poly, 1)

# --- BOUCLE DE JEU ---
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        
        if event.type == pygame.KEYDOWN:
            nx, ny = perso_pos[0], perso_pos[1]
            if event.key == pygame.K_UP:    ny -= 1
            if event.key == pygame.K_DOWN:  ny += 1
            if event.key == pygame.K_LEFT:  nx -= 1
            if event.key == pygame.K_RIGHT: nx += 1
            
            # Collision simple
            if labirynthe[ny][nx] == 0:
                perso_pos = [nx, ny]

    # --- DESSIN ---
    fenetre.fill(CIEL)

    # On dessine la carte ligne par ligne
    for r, ligne in enumerate(labirynthe):
        for c, cellule in enumerate(ligne):
            draw_block(c, r, cellule == 1)
            
            # Dessiner le joueur s'il est sur cette tuile
            if c == perso_pos[0] and r == perso_pos[1]:
                px, py = get_screen_pos(c, r)
                # Ombre
                pygame.draw.ellipse(fenetre, (20, 20, 20), (px+15, py+10, 30, 15))
                # Corps (une bille en relief)
                pygame.draw.circle(fenetre, JOUEUR, (px + TW//2, py + TH//2), 12)
                pygame.draw.circle(fenetre, (255, 120, 100), (px + TW//2 - 4, py + TH//2 - 4), 4)

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

In [ ]:
import pygame


pygame.init()
L, H = 800, 600
fenetre = pygame.display.set_mode((L, H))
pygame.display.set_caption("Mon Donjon Isométrique - Saut !")
clock = pygame.time.Clock()

# Paramètres
TW, TH = 64, 32 
CIEL = (45, 52, 54)
HERBE = (38, 166, 91)
HERBE_BORD = (30, 130, 76)
MUR_TOP = (149, 165, 166)
MUR_SIDE = (99, 110, 114)
JOUEUR = (231, 76, 60)

labirynthe = [
    [1,1,1,1,1,1,1,1],
    [1,0,0,0,0,0,0,1],
    [1,0,1,1,0,1,0,1],
    [1,0,1,0,0,0,0,1],
    [1,0,0,0,1,1,0,1],
    [1,1,1,1,1,1,1,1]
]

perso_pos = [1, 1]

z_pos = 0          # Hauteur actuelle du perso
z_velocite = 0     # Vitesse du saut
gravite = 0.8      # Force qui ramène au sol
est_en_train_de_sauter = False

def get_screen_pos(gx, gy):
    sx = (gx - gy) * (TW // 2) + (L // 2) - (TW // 2)
    sy = (gx + gy) * (TH // 2) + (H // 4)
    return sx, sy

def draw_block(x, y, is_wall):
    sx, sy = get_screen_pos(x, y)
    top_poly = [(sx + TW//2, sy), (sx + TW, sy + TH//2), (sx + TW//2, sy + TH), (sx, sy + TH//2)]
    if is_wall:
        pygame.draw.polygon(fenetre, MUR_SIDE, [top_poly[1], top_poly[2], (sx+TW, sy+TH//2+20), (sx+TW//2, sy+TH+20)])
        pygame.draw.polygon(fenetre, (45, 52, 54), [top_poly[3], top_poly[2], (sx, sy+TH//2+20), (sx+TW//2, sy+TH+20)])
        pygame.draw.polygon(fenetre, MUR_TOP, top_poly)
    else:
        pygame.draw.polygon(fenetre, HERBE, top_poly)
        pygame.draw.polygon(fenetre, HERBE_BORD, top_poly, 1)


running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        
        if event.type == pygame.KEYDOWN:
            nx, ny = perso_pos[0], perso_pos[1]
            if event.key == pygame.K_UP:    ny -= 1
            if event.key == pygame.K_DOWN:  ny += 1
            if event.key == pygame.K_LEFT:  nx -= 1
            if event.key == pygame.K_RIGHT: nx += 1
            
            # Déclenchement du saut
            if event.key == pygame.K_SPACE and not est_en_train_de_sauter:
                z_velocite = -10 # Impulsion vers le haut
                est_en_train_de_sauter = True

            if 0 <= ny < len(labirynthe) and 0 <= nx < len(labirynthe[0]):
                if labirynthe[ny][nx] == 0:
                    perso_pos = [nx, ny]

    if est_en_train_de_sauter:
        z_pos += z_velocite
        z_velocite += gravite
        if z_pos >= 0: # On a touché le sol
            z_pos = 0
            z_velocite = 0
            est_en_train_de_sauter = False

    fenetre.fill(CIEL)

    for r, ligne in enumerate(labirynthe):
        for c, cellule in enumerate(ligne):
            draw_block(c, r, cellule == 1)
            
            if c == perso_pos[0] and r == perso_pos[1]:
                px, py = get_screen_pos(c, r)
                # L'ombre reste au sol (py)
                pygame.draw.ellipse(fenetre, (20, 20, 20), (px+15, py+10, 30, 15))
                # Le corps est décalé par z_pos (qui est négatif quand on est en l'air)
                pygame.draw.circle(fenetre, JOUEUR, (px + TW//2, py + TH//2 + int(z_pos)), 12)
                pygame.draw.circle(fenetre, (255, 120, 100), (px + TW//2 - 4, py + TH//2 + int(z_pos) - 4), 4)

    pygame.display.flip()
    clock.tick(60)

pygame.quit()

In [ ]:
import pygame
import sys

# --- INITIALISATION ---
pygame.init()
L, H = 800, 600
fenetre = pygame.display.set_mode((L, H))
pygame.display.set_caption("Mon Jeu Isométrique - Version Sprite PNG")
clock = pygame.time.Clock()

# Paramètres de la grille
TW, TH = 64, 32 

# Couleurs
CIEL = (45, 52, 54)
HERBE = (38, 166, 91)
HERBE_BORD = (30, 130, 76)
MUR_TOP = (149, 165, 166)
MUR_SIDE = (99, 110, 114)

# --- CHARGEMENT DU SPRITE ---
try:
    image_originale = pygame.image.load("perso.png").convert_alpha()
    
    # ICI TU CHOISIS LA TAILLE (Largeur, Hauteur)
    # 40x60 c'est pas mal pour la grille actuelle
    sprite_perso = pygame.transform.scale(image_originale, (40, 60))
    
except:
    print("Erreur : Fichier perso.png introuvable !")
    sprite_perso = pygame.Surface((32, 48))
    sprite_perso.fill((255, 0, 0))

# --- LA CARTE ---
labirynthe = [
    [1,1,1,1,1,1,1,1],
    [1,0,0,0,0,0,0,1],
    [1,0,1,1,0,1,0,1],
    [1,0,1,0,0,0,0,1],
    [1,0,0,0,1,1,0,1],
    [1,1,1,1,1,1,1,1]
]

perso_pos = [1, 1]
z_pos = 0          
z_velocite = 0     
gravite = 0.8      
est_en_train_de_sauter = False

def get_screen_pos(gx, gy):
    sx = (gx - gy) * (TW // 2) + (L // 2) - (TW // 2)
    sy = (gx + gy) * (TH // 2) + (H // 4)
    return sx, sy

def draw_block(x, y, is_wall):
    sx, sy = get_screen_pos(x, y)
    top_poly = [(sx + TW//2, sy), (sx + TW, sy + TH//2), (sx + TW//2, sy + TH), (sx, sy + TH//2)]
    if is_wall:
        pygame.draw.polygon(fenetre, MUR_SIDE, [top_poly[1], top_poly[2], (sx+TW, sy+TH//2+20), (sx+TW//2, sy+TH+20)])
        pygame.draw.polygon(fenetre, (45, 52, 54), [top_poly[3], top_poly[2], (sx, sy+TH//2+20), (sx+TW//2, sy+TH+20)])
        pygame.draw.polygon(fenetre, MUR_TOP, top_poly)
    else:
        pygame.draw.polygon(fenetre, HERBE, top_poly)
        pygame.draw.polygon(fenetre, HERBE_BORD, top_poly, 1)

# --- BOUCLE DE JEU ---
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        
        if event.type == pygame.KEYDOWN:
            nx, ny = perso_pos[0], perso_pos[1]
            if event.key == pygame.K_UP:    ny -= 1
            if event.key == pygame.K_DOWN:  ny += 1
            if event.key == pygame.K_LEFT:  nx -= 1
            if event.key == pygame.K_RIGHT: nx += 1
            
            if event.key == pygame.K_SPACE and not est_en_train_de_sauter:
                z_velocite = -10
                est_en_train_de_sauter = True

            # Collision
            if 0 <= ny < len(labirynthe) and 0 <= nx < len(labirynthe[0]):
                if labirynthe[ny][nx] == 0:
                    perso_pos = [nx, ny]

    # Physique
    if est_en_train_de_sauter:
        z_pos += z_velocite
        z_velocite += gravite
        if z_pos >= 0:
            z_pos = 0
            z_velocite = 0
            est_en_train_de_sauter = False

    # Rendu
    fenetre.fill(CIEL)

    for r, ligne in enumerate(labirynthe):
        for c, cellule in enumerate(ligne):
            draw_block(c, r, cellule == 1)
            
            if c == perso_pos[0] and r == perso_pos[1]:
                px, py = get_screen_pos(c, r)
                # Ombre
                pygame.draw.ellipse(fenetre, (20, 20, 20), (px+15, py+15, 30, 12))
                # Affichage du PNG (centré avec prise en compte du saut)
                fenetre.blit(sprite_perso, (px + (TW - sprite_perso.get_width())//2, py - sprite_perso.get_height() + TH//2 + int(z_pos)))

    pygame.display.flip()
    clock.tick(60)

pygame.quit()
sys.exit()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
